# 04. Math Patterns for Interviews

Smaller math ideas that show up often: voting, sampling, shuffling, number base conversion.

**Learning order:** BOYER-MOORE VOTING → FISHER-YATES SHUFFLE → RESERVOIR SAMPLING → BASE 26 → ROMAN NUMERALS → UGLY NUMBERS → DECISION TABLE → PRACTICE MAP

## 1. Majority element (Boyer-Moore voting)

The element that appears **more than n/2 times**. Keep a `candidate` and a `count`: equal elements add 1, different ones subtract 1 (a pair of different elements cancels). Since the majority outnumbers all others combined, it **survives**. O(n) time, O(1) space.

In [1]:
def majority_element(nums):
    candidate, count = None, 0
    for x in nums:
        if count == 0:
            candidate = x
        count += 1 if x == candidate else -1
    return candidate

print(majority_element([3, 2, 3]), majority_element([2, 2, 1, 1, 1, 2, 2]))   # 3 2

3 2


## 2. Fisher-Yates shuffle (uniform random permutation)

For `i` from the end down to 1, swap `arr[i]` with a random element from `0..i`. Every permutation is equally likely. O(n). (A common wrong shuffle swaps each element with a random index from the **whole** array: biased.)

In [2]:
import random

def fisher_yates(arr):
    arr = arr[:]
    for i in range(len(arr) - 1, 0, -1):
        j = random.randint(0, i)
        arr[i], arr[j] = arr[j], arr[i]
    return arr

random.seed(7)
shuffled = fisher_yates([1, 2, 3, 4, 5])
print(sorted(shuffled), len(shuffled))   # [1, 2, 3, 4, 5] 5

[1, 2, 3, 4, 5] 5


## 3. Reservoir sampling

Pick **one item uniformly at random from a stream** of unknown length, using O(1) memory. For the `i`-th item (1-indexed), replace the chosen one with probability `1 / i`. Each item ends up chosen with probability `1/n`.

Real life: pick a random line from a huge log file you can read only once.

In [3]:
def reservoir_pick(stream):
    chosen = None
    for i, item in enumerate(stream, start=1):
        if random.randint(1, i) == 1:               # probability 1/i
            chosen = item
    return chosen

random.seed(1)
counts = {x: 0 for x in range(5)}
for _ in range(10000):
    counts[reservoir_pick(range(5))] += 1
print(all(1800 < c < 2200 for c in counts.values()))   # True  (each about 2000)

True


## 4. Excel sheet column title (base 26 without zero)

`1 → A`, `26 → Z`, `27 → AA`. There is no zero digit, so subtract 1 before each division: `(n - 1) % 26` gives the letter, `(n - 1) // 26` is the rest.

In [4]:
def column_title(n):
    res = []
    while n:
        n -= 1
        res.append(chr(ord("A") + n % 26))
        n //= 26
    return "".join(reversed(res))

def column_number(title):
    n = 0
    for ch in title:
        n = n * 26 + (ord(ch) - ord("A") + 1)
    return n

print(column_title(1), column_title(28), column_title(701), column_number("ZY"))   # A AB ZY 701

A AB ZY 701


## 5. Roman numerals

Values: `I=1 V=5 X=10 L=50 C=100 D=500 M=1000`. A smaller value **before** a bigger one is subtracted (`IV = 4`). For int → Roman, greedily use the largest symbol, including the subtractive pairs (`CM=900, CD=400, XC=90, XL=40, IX=9, IV=4`).

In [5]:
def roman_to_int(s):
    value = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    total = 0
    for i, ch in enumerate(s):
        if i + 1 < len(s) and value[ch] < value[s[i + 1]]:
            total -= value[ch]
        else:
            total += value[ch]
    return total

def int_to_roman(num):
    pairs = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
             (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
    res = []
    for val, sym in pairs:
        while num >= val:
            res.append(sym)
            num -= val
    return "".join(res)

print(roman_to_int("MCMXCIV"), int_to_roman(1994), int_to_roman(58))   # 1994 MCMXCIV LVIII

1994 MCMXCIV LVIII


## 6. Ugly Number II (three pointers DP)

Ugly numbers have only prime factors 2, 3, 5: `1, 2, 3, 4, 5, 6, 8, 9, 10, 12, ...`. Every ugly number is a smaller ugly number times 2, 3 or 5. Keep **one pointer per factor** into the list; the next number is the smallest candidate; advance the pointers that produced it (skips duplicates). O(n).

In [6]:
def nth_ugly_number(n):
    ugly = [1]
    i2 = i3 = i5 = 0
    while len(ugly) < n:
        nxt = min(ugly[i2] * 2, ugly[i3] * 3, ugly[i5] * 5)
        ugly.append(nxt)
        if nxt == ugly[i2] * 2: i2 += 1
        if nxt == ugly[i3] * 3: i3 += 1
        if nxt == ugly[i5] * 5: i5 += 1
    return ugly[n - 1]

print(nth_ugly_number(10), nth_ugly_number(1))   # 12 1

12 1


## 7. Decision table

| Question looks like | Tool |
|---|---|
| element appearing more than n/2 times | Boyer-Moore voting |
| random pick from a stream / equal probability | reservoir sampling |
| random shuffle | Fisher-Yates |
| column title, base conversion without zero | `n -= 1` then `% base` |
| huge answers, "mod 10⁹+7" | modular arithmetic notebook |
| count paths / arrangements | combinatorics notebook |
| primes, divisors | sieve / √n loops |
| numbers with prime-factor restrictions | multiple pointers or heap |

## 8. Practice map (for later)

| Level | Problems |
|---|---|
| Basic | Palindrome Number (9), Roman to Integer (13), Excel Sheet Column Number (171), Power of Three (326), Count Primes (204) |
| Pattern | Majority Element (169), Reverse Integer (7), Pow(x, n) (50), Sqrt(x) (69), Excel Sheet Column Title (168), Integer to Roman (12) |
| Standard | Ugly Number II (264), Unique Binary Search Trees (96), Random Pick with Weight (528), Shuffle an Array (384), Linked List Random Node (382), Super Pow (372) |
| Hard | Number of Digit One (233), Count Primes variants, Largest Divisible Subset (368), Max Points on a Line (149) |

## Quick Revision

- Boyer-Moore: candidate + count, majority survives.
- Fisher-Yates: swap with a random index in `0..i`.
- Reservoir: replace with probability `1/i`.
- Base 26 without zero: `n -= 1`.
- Ugly numbers: one pointer per prime factor.